# AIMM Data Processing

In [ ]:
from pyspark.sql import functions as F

kusto_cluster = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"
kusto_db = "pi-realtime-db"

# --- 1. AIMM Work Requests ---
df_wr = (spark.read
    .format("com.microsoft.kusto.spark.synapse.datasource")
    .option("spark.synapse.linkedService", "pi-realtime-eventhouse")
    .option("kustoCluster", kusto_cluster)
    .option("kustoDatabase", kusto_db)
    .option("kustoQuery", "AimmWorkRequests")
    .load()
)
# Cast entity_identity to string to match dim_equipment.icare_id
df_wr = df_wr.withColumn("entity_identity", F.col("entity_identity").cast("string"))
print(f"Work Requests loaded: {df_wr.count()}")

# --- 2. Write to dbo + gold ---
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
df_wr.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("fact_work_requests")
df_wr.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gold.fact_work_requests")
print("Written: fact_work_requests (dbo + gold)")

# --- Summary ---
print(f"\nfact_work_requests: {df_wr.count()} rows")
df_wr.groupBy("wr_status").agg(F.count("*").alias("n")).orderBy(F.desc("n")).show(10)